In [2]:
import sys
sys.path.append("..")
import importlib
from pathlib import Path
import pandas as pd
import src.data as data
importlib.reload(data)

S0 = data.get_spot("SPY")
r = data.get_risk_free_rate()
print(f"Spot SPY = {S0:.2f} | taux sans risque = {r:.2%}")

raw = data.download_option_chain("SPY")
clean = data.clean_chain(raw)
print(f"{len(raw)} options téléchargées, {len(clean)} après nettoyage")
print(f"{clean['expiry'].nunique()} maturités, de {clean['T'].min()*365:.0f} à {clean['T'].max()*365:.0f} jours")

# Sauvegarde de la photo du marché
today = pd.Timestamp.today().strftime("%Y-%m-%d")
Path("../data").mkdir(exist_ok=True)
clean.to_csv(f"../data/spy_options_{today}.csv", index=False)
pd.Series({"date": today, "spot": S0, "r": r}).to_csv(f"../data/spy_market_{today}.csv")

clean.head(10)

Spot SPY = 779.56 | taux sans risque = 4.03%
10014 options téléchargées, 8082 après nettoyage
26 maturités, de 7 à 836 jours


,type,expiry,T,strike,bid,ask,mid,lastPrice,volume,openInterest,impliedVolatility
0,call,2026-10-13,0.019178,710.0,69.57,71.69,70.630,70.55,48.0,1,0.469244
1,call,2026-10-13,0.019178,730.0,49.62,51.74,50.680,50.62,30.0,1,0.363532
2,call,2026-10-13,0.019178,734.0,45.63,47.75,46.690,46.70,54.0,3,0.341926
3,call,2026-10-13,0.019178,735.0,44.64,46.80,45.720,45.69,36.0,1,0.338569
4,call,2026-10-13,0.019178,736.0,43.64,45.80,44.720,44.86,20.0,3,0.333014
5,call,2026-10-13,0.019178,739.0,40.66,42.77,41.715,41.84,74.0,3,0.314826
6,call,2026-10-13,0.019178,740.0,39.66,41.78,40.720,40.94,42.0,4,0.309638
7,call,2026-10-13,0.019178,742.0,37.67,39.83,38.750,38.99,36.0,3,0.300361
8,call,2026-10-13,0.019178,743.0,36.68,38.84,37.760,37.77,32.0,6,0.295051
9,call,2026-10-13,0.019178,745.0,35.43,36.25,35.840,35.77,78.0,201,0.258735


In [3]:
import src.black_scholes as bs
import src.implied_vol as iv
importlib.reload(bs)
importlib.reload(iv)

print("Test aller-retour : prix BS → vol implicite")
for sigma_true in [0.10, 0.20, 0.50, 1.00]:
    for K_test in [80, 100, 120]:
        for opt in ["call", "put"]:
            p = bs.bs_price(100, K_test, 1, 0.05, sigma_true, opt)
            s = iv.implied_vol(p, 100, K_test, 1, 0.05, opt)
            ok = "✅" if abs(s - sigma_true) < 1e-6 else "❌"
            print(f"{ok} σ vraie = {sigma_true:.2f} | K = {K_test} | {opt:<4} | σ retrouvée = {s:.6f}")

Test aller-retour : prix BS → vol implicite
✅ σ vraie = 0.10 | K = 80 | call | σ retrouvée = 0.100000
✅ σ vraie = 0.10 | K = 80 | put  | σ retrouvée = 0.100000
✅ σ vraie = 0.10 | K = 100 | call | σ retrouvée = 0.100000
✅ σ vraie = 0.10 | K = 100 | put  | σ retrouvée = 0.100000
✅ σ vraie = 0.10 | K = 120 | call | σ retrouvée = 0.100000
✅ σ vraie = 0.10 | K = 120 | put  | σ retrouvée = 0.100000
✅ σ vraie = 0.20 | K = 80 | call | σ retrouvée = 0.200000
✅ σ vraie = 0.20 | K = 80 | put  | σ retrouvée = 0.200000
✅ σ vraie = 0.20 | K = 100 | call | σ retrouvée = 0.200000
✅ σ vraie = 0.20 | K = 100 | put  | σ retrouvée = 0.200000
✅ σ vraie = 0.20 | K = 120 | call | σ retrouvée = 0.200000
✅ σ vraie = 0.20 | K = 120 | put  | σ retrouvée = 0.200000
✅ σ vraie = 0.50 | K = 80 | call | σ retrouvée = 0.500000
✅ σ vraie = 0.50 | K = 80 | put  | σ retrouvée = 0.500000
✅ σ vraie = 0.50 | K = 100 | call | σ retrouvée = 0.500000
✅ σ vraie = 0.50 | K = 100 | put  | σ retrouvée = 0.500000
✅ σ vraie = 0.50 |

In [4]:
import src.vol_surface as vs
importlib.reload(iv)
importlib.reload(vs)

smiles = vs.compute_smiles(clean, S0, r)
print(f"{len(smiles)} vols implicites calculées sur {smiles['expiry'].nunique()} maturités\n")
print(smiles.groupby("expiry")[["T", "F", "q"]].first().round(4))

4803 vols implicites calculées sur 26 maturités

                 T         F       q
expiry                              
2026-10-13  0.0192  780.5454 -0.0255
2026-10-14  0.0219  780.6306 -0.0223
2026-10-15  0.0247  780.6957 -0.0187
2026-10-16  0.0274  780.8849 -0.0216
2026-10-23  0.0466  781.4258 -0.0110
2026-10-30  0.0658  782.0602 -0.0083
2026-11-06  0.0849  782.7291 -0.0074
2026-11-13  0.1041  783.2561 -0.0051
2026-11-20  0.1233  783.8141 -0.0038
2026-11-30  0.1507  784.4970 -0.0015
2026-12-18  0.2000  785.4234  0.0029
2026-12-31  0.2356  785.5956  0.0076
2027-01-15  0.2767  787.0983  0.0056
2027-01-29  0.3151  788.1871  0.0054
2027-02-26  0.3918  790.7105  0.0041
2027-03-19  0.4493  792.4032  0.0040
2027-03-31  0.4822  792.4662  0.0063
2027-06-17  0.6959  799.3418  0.0043
2027-06-30  0.7315  799.6807  0.0055
2027-09-17  0.9479  807.0053  0.0038
2027-09-30  0.9836  807.3255  0.0048
2027-12-17  1.1973  813.9243  0.0043
2028-01-21  1.2932  816.3538  0.0047
2028-06-16  1.6959  828.53